# BirdCLEF 2026 — Ensemble Submission (ConvNeXt-Small V3 + ECA-NFNet-L0), weighted

Weighted average of sigmoid outputs from two models (both trained on v1 mel params, hop=512):
- **Model A**: `multilabel_234_v3/3` — ConvNeXt-Small (+ R2 pseudo-labels), weight 0.6
- **Model B**: `multilabel_234/9` — ECA-NFNet-L0, weight 0.4

In [ ]:
import json
import warnings
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torchvision import transforms as T
from fastai.vision.all import load_learner

warnings.filterwarnings('ignore', category=UserWarning, module='fastai')

In [ ]:
import kagglehub
path = kagglehub.competition_download('birdclef-2026')
print('Path to competition files:', path)

In [ ]:
# Model A — ConvNeXt-Small V3 (+ R2 pseudo-labels), v1 mel params (hop=512, librosa defaults)
MODEL_A_PATH = '/kaggle/input/models/ucheozoemena/bird-clef-classifier/pytorch/multilabel_234_v3/3/model_multilabel_234.pkl'
MODEL_A_VOCAB_PATH = '/kaggle/input/models/ucheozoemena/bird-clef-classifier/pytorch/multilabel_234_v3/3/vocab.json'

# Model B — ECA-NFNet-L0, v1 mel params (hop=512, librosa defaults)
MODEL_B_PATH = '/kaggle/input/models/ucheozoemena/bird-clef-classifier/pytorch/multilabel_234/9/model_multilabel_234.pkl'
MODEL_B_VOCAB_PATH = '/kaggle/input/models/ucheozoemena/bird-clef-classifier/pytorch/multilabel_234/9/vocab.json'

WEIGHT_A = 0.6  # ConvNeXt-Small V3 — stronger solo model (0.805)
WEIGHT_B = 0.4  # ECA-NFNet-L0

HOP_LENGTH = 512  # shared mel param for both models

TEST_DIR       = Path(path) / 'test_soundscapes'
TARGET_SIZE    = (224, 224)
CLIP_DURATION  = 5
SAMPLE_RATE    = 32000
BATCH_SIZE     = 64
STRIDE_DURATION = 2.5

In [ ]:
sample_sub = pd.read_csv(Path(path) / 'sample_submission.csv')
all_species = [c for c in sample_sub.columns if c != 'row_id']
assert len(all_species) == 234, f'expected 234, got {len(all_species)}'

learn_a = load_learner(MODEL_A_PATH, cpu=True)
learn_b = load_learner(MODEL_B_PATH, cpu=True)

for label, learn, vocab_path in [
    ('A (ConvNeXt-Small)', learn_a, MODEL_A_VOCAB_PATH),
    ('B (ECA-NFNet-L0)',   learn_b, MODEL_B_VOCAB_PATH),
]:
    vocab = list(learn.dls.vocab)
    assert vocab == all_species, f'{label} vocab does not match sample_submission column order'
    if Path(vocab_path).exists():
        assert json.load(open(vocab_path)) == vocab, f'{label} vocab.json disagrees with learner'
    print(f'{label} vocab ok ({len(vocab)} classes)')

learn_a.model.eval()
learn_b.model.eval()
device = next(learn_a.model.parameters()).device
print(f'Running on: {device}')

In [ ]:
from scipy.ndimage import convolve1d

clip_length    = CLIP_DURATION * SAMPLE_RATE
stride_samples = int(STRIDE_DURATION * SAMPLE_RATE)

tfm = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def window_to_img(window):
    lo, hi = float(window.min()), float(window.max())
    arr = np.zeros_like(window, dtype=np.uint8) if hi == lo else (
        (window - lo) / (hi - lo) * 255
    ).astype(np.uint8)
    return Image.fromarray(arr).resize(TARGET_SIZE).convert('RGB')


def mel_to_submission_preds(learn, S_db, hop_length, n_samples):
    """Run model over all overlapping clips; aggregate into per-5s-window max predictions."""
    frames_per_clip = int(CLIP_DURATION * SAMPLE_RATE / hop_length)
    stride_frames   = int(STRIDE_DURATION * SAMPLE_RATE / hop_length)

    imgs = []
    k = 0
    while k * stride_samples + clip_length <= n_samples:
        window = S_db[:, k * stride_frames : k * stride_frames + frames_per_clip]
        imgs.append(window_to_img(window))
        k += 1
    n_overlapping = len(imgs)

    overlap_preds = []
    for i in range(0, n_overlapping, BATCH_SIZE):
        batch = torch.stack([tfm(img) for img in imgs[i:i + BATCH_SIZE]]).to(device)
        with torch.no_grad():
            logits = learn.model(batch)
        overlap_preds.append(torch.sigmoid(logits).cpu().numpy())
    overlap_preds = np.vstack(overlap_preds)  # (n_overlapping, 234)

    n_submission = n_samples // clip_length
    window_preds = []
    for j in range(n_submission):
        k_lo = max(0, 2 * j - 1)
        k_hi = min(n_overlapping - 1, 2 * j + 1)
        window_preds.append(overlap_preds[k_lo:k_hi + 1].max(axis=0))
    return np.array(window_preds)  # (n_submission, 234)


def temporal_smooth(preds, weights=(0.1, 0.2, 0.4, 0.2, 0.1)):
    """Blend each 5s window with a weighted contribution from its neighbours.
    Handles boundary windows with 'nearest' padding so edge clips aren't under-smoothed."""
    kernel = np.array(weights, dtype=np.float32)
    kernel /= kernel.sum()
    return convolve1d(preds, kernel, axis=0, mode='nearest')

In [ ]:
test_files = sorted(TEST_DIR.glob('*.ogg'))
if not test_files:
    fallback_dir = Path(path) / 'train_soundscapes'
    test_files = sorted(fallback_dir.glob('*.ogg'))[:3]
    print(f'[dry-run] test_soundscapes empty, using {len(test_files)} train_soundscapes files')
else:
    print(f'Found {len(test_files)} test soundscape files')

row_ids   = []
all_preds = []

for soundscape in test_files:
    samples, _ = librosa.load(soundscape, sr=SAMPLE_RATE)
    n_samples    = len(samples)
    n_submission = n_samples // clip_length

    S_db = librosa.power_to_db(
        librosa.feature.melspectrogram(y=samples, sr=SAMPLE_RATE, hop_length=HOP_LENGTH),
        ref=np.max,
    )
    preds_a = mel_to_submission_preds(learn_a, S_db, HOP_LENGTH, n_samples)
    preds_b = mel_to_submission_preds(learn_b, S_db, HOP_LENGTH, n_samples)

    preds_ensemble = WEIGHT_A * preds_a + WEIGHT_B * preds_b
    preds_ensemble = temporal_smooth(preds_ensemble)
    all_preds.append(preds_ensemble)
    for j in range(n_submission):
        row_ids.append(f'{soundscape.stem}_{(j + 1) * CLIP_DURATION}')

preds_np = np.vstack(all_preds)
assert preds_np.shape == (len(row_ids), 234), preds_np.shape
print(f'Inference complete: {preds_np.shape[0]} clips across {len(test_files)} files')

In [ ]:
submission = pd.DataFrame(preds_np, columns=all_species)
submission.insert(0, 'row_id', row_ids)
assert list(submission.columns) == ['row_id'] + all_species
submission.to_csv('submission.csv', index=False)
print(f'Done. {len(submission)} rows written.')
submission.head()